# Comparisons and Boolean masks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/05-boolean-masks.ipynb)

Many business questions start with a condition. Which products cost more than $50? How many customers visit more
than six times a month, and what share of all customers is that? In this chapter, we answer such questions with
NumPy. A comparison on an array gives an array of `True` and `False` values, and that array lets us choose, count
and summarize values without writing a loop.

**In this chapter, you will learn to:**

- compare an array with a number, and get an array of `True` and `False` values: a **Boolean mask**;
- keep only the values you want with a mask, as in `prices[prices > 50]`;
- combine conditions with `&` (and), `|` (or) and `~` (not), and know why the parentheses are needed;
- summarize an array with `sum`, `mean`, `min`, `max` and `np.median`, and find the position of the largest value
  with `argmax`;
- count the values that meet a condition with `mask.sum()`, and compute their share with `mask.mean()`.

In [ ]:
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

We use NumPy in the whole chapter. pandas only reads a data file, near the end.

## A comparison gives True or False for each value

An online shop sells eight products. Here are their prices, in dollars, and the units of each product that the
shop sold last week, in the same order:

In [ ]:
prices = np.array([12.5, 48.0, 75.0, 19.99, 120.0, 55.5, 8.75, 64.0])
units = np.array([30, 12, 5, 41, 2, 9, 60, 15])
prices > 50

> **In Thonny:** wrap the last line in `print()`: `print(prices > 50)`. `print()` shows the same values without
> `array(...)` and without commas: `[False False  True False  True  True False  True]`.

Arithmetic on an array, such as `prices * 2`, works on every value at once. A comparison works the same way:
`prices > 50` compares **every** price with 50, and gives one answer for each price. The first price, 12.5, is not
above 50, so the first answer is `False`. The third price, 75.0, is above 50, so the third answer is `True`.

`True` and `False` are Python's two **Boolean** values (named after the mathematician George Boole). Let's keep the
answers in a variable, and look at its dtype:

In [ ]:
expensive = prices > 50
expensive.dtype

The dtype is `bool`: the array holds Boolean values. Such an array is called a **Boolean mask**, or simply a
**mask**. It has one `True` or `False` for each product, in the same order as `prices`. Like a painter's mask, it
covers some values and lets the others through, as we see next.

All of Python's comparison operators work on arrays: `>`, `>=`, `<`, `<=`, `==` (equal) and `!=` (not equal).

## Filtering: keeping the values where the mask is True

Put a mask in square brackets after an array, and NumPy keeps the values where the mask is `True`:

In [ ]:
prices[expensive]

Four of the eight prices are above $50 (NumPy shows `75.` for 75.0). This is called **filtering**: the mask chooses
the values, and the result is a new, shorter array. The array `prices` itself does not change.

Usually, we write the comparison directly inside the brackets: `prices[prices > 50]`, which reads "prices, where
prices are above 50". And the mask can choose values from **another** array of the same length. The positions
match: the first value of `units` belongs to the first product, and so on. So how many units did the expensive
products sell?

In [ ]:
units[prices > 50]

The four products above $50 sold only 5, 2, 9 and 15 units. The mask was made from `prices`, but it chooses values
from `units`. A condition on one column chooses the values of another column: that is also how pandas filters the
rows of a table, in the chapter *Filtering rows*.

> **Try it:** change `50` to `100` in the cell above, and run it again. Which units remain?

## Combining conditions with &, | and ~

Which products cost between $20 and $70? Two conditions must both be true. For masks, `&` means "and":

In [ ]:
prices[(prices > 20) & (prices < 70)]

`&` combines the two masks position by position: the result is `True` only where **both** masks are `True`. The
price 19.99 is not above 20, and 75.0 is not below 70, so they are left out.

Two more operators work the same way. `|` (the vertical bar) means "or": `True` where **at least one** of the masks
is `True`. `~` (the tilde) means "not": it turns every `True` into `False`, and every `False` into `True`. Here are
the very cheap or very expensive products, and then the products that are *not* expensive:

In [ ]:
print(prices[(prices < 10) | (prices > 100)])
print(prices[~expensive])

$8.75 is below 10 and $120 is above 100. The second line gives the same prices as `prices[prices <= 50]`. In short:

- `a & b` (and) is `True` where both `a` and `b` are `True`;
- `a | b` (or) is `True` where at least one of them is `True`;
- `~a` (not) is `True` where `a` is `False`.

## Why not `and`? Why the parentheses?

Python's words `and`, `or` and `not` work on **one** `True` or `False`, as in an `if` statement. They do not work
on a whole mask:

In [ ]:
prices[(prices > 20) and (prices < 70)]

The last line says `ValueError: The truth value of an array with more than one element is ambiguous`. To use
`and`, Python must know whether the whole mask `prices > 20` is true or false. But some of its values are `True`
and some are `False`, so NumPy refuses to guess ("ambiguous" means that it could mean more than one thing). The
message suggests `a.any()` (is *any* value `True`?) or `a.all()` (are *all* values `True`?), but that is not what we
want here. For masks, use `&`, `|` and `~` instead of `and`, `or` and `not`.

The parentheses matter too, because Python computes `&` **before** `>` and `<`. Without parentheses,
`prices > 20 & prices < 70` starts with `20 & prices`, which is not what we mean, and it gives an error.

> **Common mistake:** leaving out the parentheses, as in `prices[prices > 20 & prices < 70]`. Put each comparison
> in its own parentheses: `prices[(prices > 20) & (prices < 70)]`.

## Summaries: one number from many values

A manager often wants one number instead of eight: the total, the average, the smallest, the largest. Arrays have
methods for these. We print them, to see several values from one cell:

In [ ]:
print("total:  ", units.sum())
print("average:", units.mean())
print("lowest: ", units.min())
print("highest:", units.max())
print("median: ", np.median(units))

The shop sold 174 units in all, 21.75 per product on average. The fewest units of one product were 2, and the most
were 60. (Without `print()`, Colab shows such a result as `np.int64(174)` or `np.float64(21.75)`, as in the
chapter *Indexing, slicing and arithmetic*: it is simply the number.)

The **median** is the middle value: sort the values, and take the one in the middle. With an even number of
values, two values are in the middle, and the median is their average. Sorted, our units are 2, 5, 9, 12, 15, 30,
41, 60. The middle two are 12 and 15, so the median is 13.5: half of the products sold fewer than 13.5 units, and
half sold more.

Why is the average (21.75) so much bigger than the median (13.5)? Two products sold a lot (41 and 60 units), and
they pull the average up. The median does not care how big the biggest values are. When a few values are very
large, as with sales or incomes, the median often describes a typical value better than the average.

> **Common mistake:** writing `units.median()`. Arrays have no `median` method, so this gives an
> `AttributeError`. Write `np.median(units)`: the median is a NumPy *function*.

## Where is the largest value? argmax

`max()` gives the largest value, but not *which* product it belongs to. `argmax()` gives the **position** of the
largest value:

In [ ]:
prices.argmax()

The most expensive product is at position 4: the fifth product, because positions start at 0. With the position,
we can look up anything else about that product, in any list or array in the same order. Here are the names of
the products, as a Python list:

In [ ]:
products = ["mug", "backpack", "desk lamp", "notebooks", "headphones", "keyboard", "pens", "stand"]
top = prices.argmax()
print(products[top], "cost", prices[top], "dollars and sold", units[top], "units")

The most expensive product, the headphones, sold only 2 units last week. There is also `argmin()`, the position of
the smallest value. If the largest value appears more than once, `argmax()` gives the first of its positions.

## Counting with a mask

How many products cost more than $50, and what share of all products is that? We could count the `True` values by
eye, but NumPy can do it for us:

In [ ]:
print("count:", expensive.sum())
print("share:", expensive.mean())

In arithmetic, `True` counts as 1 and `False` as 0. So **the sum of a mask is the number of `True` values**: 4
products cost more than $50. And **the mean of a mask is their share**: 4 out of 8 is 0.5, so 50% of the products
cost more than $50. (Multiply a share by 100 to get a percentage.)

## 500 real customers

Eight products are easy to check by eye. Let's use the same tools on 500 values: the customers of the chapter
*A first machine-learning example*. For each member of a store chain's loyalty program, `visits_per_month` is how
often the member visits a store, on average, each month.

In [ ]:
customers = pd.read_csv(DATA_URL + "customers.csv")
visits = customers["visits_per_month"].to_numpy()
visits[:10]

The first line reads the data file into a table. The second line takes the column `visits_per_month` from the
table, and `.to_numpy()` turns it into a NumPy array. The chapter *Series and DataFrames* explains these two
lines. For now, all we need is that `visits` is an ordinary NumPy array. The slice `[:10]` shows its first ten
values: the first customer visits 5.6 times a month, the fourth 11.9 times.

What does a typical customer do?

In [ ]:
print("customers:", visits.size)
print("average:  ", np.round(visits.mean(), 2))
print("median:   ", np.median(visits))
print("lowest:   ", visits.min())
print("highest:  ", visits.max())

There are 500 customers. `np.round(..., 2)` rounds the average to two decimals: 3.65 visits a month. The median is
3.3, a little lower than the average: a few customers visit very often, up to 12 times a month, and they pull the
average up.

> **Note:** two columns of `customers.csv`, `age` and `avg_basket`, have missing values. In a NumPy array, a
> missing value is `nan` ("not a number"), and the sum or the mean of an array with a `nan` in it is `nan` too. That
> is why this chapter uses `visits_per_month` and `annual_spend`, which have no missing values. The chapter
> *Preparing data for machine learning* shows what to do with missing values.

## Counting and comparing groups of customers

The marketing team calls the customers who visit more than six times a month *frequent visitors*. How many are
there, and what share of all customers are they?

In [ ]:
frequent = visits > 6
print("frequent visitors:", frequent.sum())
print("share:", frequent.mean())

64 of the 500 customers are frequent visitors: a share of 0.128, or 12.8%.

Do frequent visitors spend more? The column `annual_spend` holds what each customer spent last year, in dollars,
in the same order as `visits`. We turn it into an array too, and use the mask `frequent` (made from `visits`) to
choose values from `spend`:

In [ ]:
spend = customers["annual_spend"].to_numpy()
print("frequent visitors:", np.round(spend[frequent].mean(), 2))
print("other customers:  ", np.round(spend[~frequent].mean(), 2))

Frequent visitors spent $5,985.77 on average, more than two and a half times the $2,326.06 of the other customers.
With one mask, and `~` for the other group, we compared two groups of customers in two lines. The models of the
chapter *A first machine-learning example* learned exactly this kind of connection between visits and spending.

> **Try it:** change `6` to `3` in `frequent = visits > 6`, and run the two cells again. How do the count, the
> share and the two averages change?

## Summary

- A comparison on an array, such as `prices > 50`, compares every value and gives an array of `True` and `False`
  values (dtype `bool`): a **Boolean mask**.
- `a[mask]` keeps the values of `a` where the mask is `True`: `prices[prices > 50]`. The mask may come from
  another array of the same length: `units[prices > 50]`.
- Combine masks with `&` (and), `|` (or) and `~` (not), and put each comparison in parentheses:
  `(prices > 20) & (prices < 70)`. The words `and`, `or` and `not` give a `ValueError` on arrays.
- Summaries: `a.sum()`, `a.mean()`, `a.min()`, `a.max()`, `np.median(a)`. `a.argmax()` and `a.argmin()` give the
  position of the largest and the smallest value.
- `True` counts as 1: `mask.sum()` counts the values that meet a condition, and `mask.mean()` gives their share.

## Practice questions

### Question 1 (Easy)

What does `np.array([4, 9, 2, 7]) >= 7` give? What is the dtype of the result? Check your answer with Python.

<details class="yr-answer">
<summary>Answer</summary>

It compares each value with 7 and gives `array([False,  True, False,  True])`: 9 and 7 are at least 7. The dtype
is `bool`.

```python
check = np.array([4, 9, 2, 7]) >= 7
print(check)
print(check.dtype)
```

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

Show the units sold of the products that cost less than $20. What do you notice?

<details class="yr-answer">
<summary>Answer</summary>

Make the mask from `prices`, and use it on `units`. The three cheap products sold 30, 41 and 60 units: many more
than the expensive ones.

```python
units[prices < 20]
```

</details>

In [ ]:
# Write your answer here

### Question 3 (Easy)

How many customers visit a store **less than once** a month? What percentage of all customers is that?

<details class="yr-answer">
<summary>Answer</summary>

13 customers, a share of 0.026, which is 2.6%. The sum of a mask counts its `True` values, and the mean gives
their share.

```python
rare = visits < 1
print(rare.sum())
print(rare.mean() * 100)
```

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

This line should show the units between 10 and 40, but it fails. Why? Fix it.

```python
units[units > 10 and units < 40]
```

<details class="yr-answer">
<summary>Answer</summary>

`and` works on one `True` or `False`, not on a whole mask, so it gives a `ValueError`. Use `&`, and put each
comparison in parentheses. The result is 30, 12 and 15.

```python
units[(units > 10) & (units < 40)]
```

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

The revenue of each product is its units times its price. What was the total revenue of the products that cost
more than $50?

<details class="yr-answer">
<summary>Answer</summary>

Compute the revenue of each product, keep the revenue of the expensive products with a mask, and add it up:
$2,074.50.

```python
revenue = units * prices
revenue[prices > 50].sum()
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Hard)

Which customer spent the most last year? Give the position, the amount, and how often that customer visits a
store.

<details class="yr-answer">
<summary>Answer</summary>

`argmax()` gives the position, 353. That customer spent $11,507.86 and visits 12 times a month, the most visits of
any customer.

```python
best = spend.argmax()
print(best, spend[best], visits[best])
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Hard)

The marketing team wants to call the frequent visitors (more than 6 visits a month) who spent **less than**
$3,000 last year. How many are there? Show their visits and their spending.

<details class="yr-answer">
<summary>Answer</summary>

Combine the two conditions with `&`. The masks come from two different arrays, of the same length. Only 4 of the
64 frequent visitors spent less than $3,000.

```python
calls = (visits > 6) & (spend < 3000)
print(calls.sum())
print(visits[calls])
print(spend[calls])
```

</details>

In [ ]:
# Write your answer here

### Question 8 (Challenge)

What share of the customers spent **more than the average** spending? Is it more or less than half? Use the
median to explain why.

<details class="yr-answer">
<summary>Answer</summary>

The share is 0.412: less than half of the customers spent more than the average of $2,794.51. The median,
$2,322.54, is lower than the average, because a few customers spent a lot (up to $11,507.86) and pull the average
up. Half of the customers spent more than the median, so fewer than half spent more than the higher average.

```python
above = spend > spend.mean()
print(above.mean())
print(np.round(spend.mean(), 2), np.median(spend))
```

</details>

In [ ]:
# Write your answer here